## Cleaning

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, accuracy_score
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from collections import Counter
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Embedding, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

jeopardy = pd.read_csv('JEOPARDY_CSV.csv') 

jeopardy

,Show Number,Air Date,Round,Category,Value,Question,Answer
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was ...",Copernicus
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,No. 2: 1912 Olympian; football star at Carlisl...,Jim Thorpe
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,The city of Yuma in this state has a record av...,Arizona
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", th...",McDonald's
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Co...",John Adams
...,...,...,...,...,...,...,...
216925,4999,2006-05-11,Double Jeopardy!,RIDDLE ME THIS,$2000,This Puccini opera turns on the solution to 3 ...,Turandot
216926,4999,2006-05-11,Double Jeopardy!,"""T"" BIRDS",$2000,In North America this term is properly applied...,a titmouse
216927,4999,2006-05-11,Double Jeopardy!,AUTHORS IN THEIR YOUTH,$2000,"In Penny Lane, where this ""Hellraiser"" grew up...",Clive Barker
216928,4999,2006-05-11,Double Jeopardy!,QUOTATIONS,$2000,"From Ft. Sill, Okla. he made the plea, Arizona...",Geronimo


In [2]:
jeopardy.columns = [s.strip() for s in jeopardy.columns];
jeopardy.columns

Index(['Show Number', 'Air Date', 'Round', 'Category', 'Value', 'Question',
       'Answer'],
      dtype='object')

In [3]:
jeopardy_data = jeopardy.copy()

jeopardy.columns = jeopardy.columns.str.strip()

jeopardy_data['Value'] = jeopardy_data['Value'].replace(r'[\$,]', '', regex=True).astype(float)
jeopardy_data['Value'] = jeopardy_data['Value'].fillna(0)

# Step 2: Remove HTML tags from the 'Question' column
jeopardy_data['Question'] = jeopardy_data['Question'].str.replace(r'<.*?>', '', regex=True)

# Step 3: Filter and clean the dataset
jeopardy_adjusted_data = jeopardy_data[
    (jeopardy_data['Value'] != 0) & (~jeopardy_data['Value'].isna())
]




In [4]:
import pandas as pd

# Ensure 'Air Date' is in datetime format
jeopardy_adjusted_data['Air Date'] = pd.to_datetime(jeopardy_adjusted_data['Air Date'])

# Apply the transformation
jeopardy_adjusted_data['Value'] = jeopardy_adjusted_data.apply(
    lambda row: row['Value'] * 2 if row['Air Date'] < pd.Timestamp('2001-11-26') else row['Value'], 
    axis=1
)



C:\Users\jACo\AppData\Local\Temp\ipykernel_12600\1423083528.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jeopardy_adjusted_data['Air Date'] = pd.to_datetime(jeopardy_adjusted_data['Air Date'])
C:\Users\jACo\AppData\Local\Temp\ipykernel_12600\1423083528.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jeopardy_adjusted_data['Value'] = jeopardy_adjusted_data.apply(


## Modeling

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

# Tokenize the 'Question' column and remove stopwords
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import os



In [6]:
jeopardy_adjusted_data['Combined_Text'] = (
    jeopardy_adjusted_data['Question'] + " " +
    jeopardy_adjusted_data['Answer'] + " " +
    jeopardy_adjusted_data['Round']
)

# Preview the updated dataset
print(jeopardy_adjusted_data[['Combined_Text']].head())

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Ensure the NLTK stopwords are downloaded
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')


# Define stop words
# stop_words = set(stopwords.words('english'))
stop_words = set()

# Define the cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ''  # Handle non-string entries
    tokens = word_tokenize(text.lower())  # Tokenize and convert to lowercase
    return ' '.join([word for word in tokens if word.isalpha() and word not in stop_words])

# Ensure 'Combined_Text' contains only strings
jeopardy_adjusted_data['Combined_Text'] = jeopardy_adjusted_data['Combined_Text'].fillna('').astype(str)

# Apply the cleaning function to the 'Combined_Text' column
jeopardy_adjusted_data['cleaned_combined_text'] = jeopardy_adjusted_data['Combined_Text'].apply(clean_text)

# Example check
print(jeopardy_adjusted_data[['Combined_Text', 'cleaned_combined_text']].head())



C:\Users\jACo\AppData\Local\Temp\ipykernel_12600\1673360197.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jeopardy_adjusted_data['Combined_Text'] = (


                                       Combined_Text
0  For the last 8 years of his life, Galileo was ...
1  No. 2: 1912 Olympian; football star at Carlisl...
2  The city of Yuma in this state has a record av...
3  In 1963, live on "The Art Linkletter Show", th...
4  Signer of the Dec. of Indep., framer of the Co...


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jACo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jACo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jACo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
C:\Users\jACo\AppData\Local\Temp\ipykernel_12600\1673360197.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jeopardy_adjusted_data['Combined_Text'] = jeopardy_adjusted_data['Combined_Text'].fillna('').astype(str)


                                       Combined_Text  \
0  For the last 8 years of his life, Galileo was ...   
1  No. 2: 1912 Olympian; football star at Carlisl...   
2  The city of Yuma in this state has a record av...   
3  In 1963, live on "The Art Linkletter Show", th...   
4  Signer of the Dec. of Indep., framer of the Co...   

                               cleaned_combined_text  
0  for the last years of his life galileo was und...  
1  no olympian football star at carlisle indian s...  
2  the city of yuma in this state has a record av...  
3  in live on the art linkletter show this compan...  
4  signer of the of framer of the constitution of...  


C:\Users\jACo\AppData\Local\Temp\ipykernel_12600\1673360197.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jeopardy_adjusted_data['cleaned_combined_text'] = jeopardy_adjusted_data['Combined_Text'].apply(clean_text)


In [7]:
# Calculate the frequency of each 'Value'
value_counts = jeopardy_adjusted_data['Value'].value_counts()

# Get the values that occur more than or equal to 1000 times
valid_values = value_counts[value_counts >= 1000].index

# Filter the DataFrame to keep only the rows where 'Value' is in the valid values list
jeopardy_adjusted_data = jeopardy_adjusted_data[jeopardy_adjusted_data['Value'].isin(valid_values)]

# Check the shape of the filtered data
jeopardy_adjusted_data['Value'].value_counts()


# Split into Jeopardy and Double Jeopardy datasets
jeopardy_data = jeopardy_adjusted_data[jeopardy_adjusted_data['Round'] == 'Jeopardy!'].copy()
double_jeopardy_data = jeopardy_adjusted_data[jeopardy_adjusted_data['Round'] == 'Double Jeopardy!'].copy()

jeopardy_data = jeopardy_data[jeopardy_data['Value'].isin(valid_values)]
double_jeopardy_data = double_jeopardy_data[double_jeopardy_data['Value'].isin(valid_values)]

# Verify the sizes of the datasets
print(f"Jeopardy dataset size: {jeopardy_data.shape}")
print(f"Double Jeopardy dataset size: {double_jeopardy_data.shape}")


Jeopardy dataset size: (106425, 9)
Double Jeopardy dataset size: (101735, 9)


In [8]:
# For the jeopardy round
jeopardy_data = jeopardy_data[~jeopardy_data['Value'].isin([2000, 1600, 1200])]

# For the double jeopardy round
double_jeopardy_data = double_jeopardy_data[~double_jeopardy_data['Value'].isin([1000, 600, 200])]

# To confirm the filtering, you can print value counts
print("Filtered Jeopardy 'Value' counts:")
print(jeopardy_data['Value'].value_counts())

print("\nFiltered Double Jeopardy 'Value' counts:")
print(double_jeopardy_data['Value'].value_counts())




Filtered Jeopardy 'Value' counts:
Value
200.0     21683
400.0     21467
1000.0    21326
600.0     20794
800.0     20267
Name: count, dtype: int64

Filtered Double Jeopardy 'Value' counts:
Value
400.0     21466
800.0     20834
2000.0    20731
1200.0    19503
1600.0    18600
Name: count, dtype: int64


In [9]:
import keras

from keras import layers

In [10]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Ensure the 'Value' column is categorical
jeopardy_data['Value'] = pd.Categorical(jeopardy_data['Value'])
jeopardy_data['Value_cat'] = jeopardy_data['Value'].cat.codes

# Define features (X) and target (y)
X_j = jeopardy_data['cleaned_combined_text']
y_j = to_categorical(jeopardy_data['Value_cat'])

# Perform train-test split
X_train_j, X_test_j, y_train_j, y_test_j = train_test_split(
    X_j, y_j, test_size=0.2, random_state=42, stratify=jeopardy_data['Value_cat']
)

# Recheck dimensions
print(f"X_train shape: {len(X_train_j)}")
print(f"y_train shape: {y_train_j.shape}")
print(f"X_test shape: {len(X_test_j)}")
print(f"y_test shape: {y_test_j.shape}")



X_train shape: 84429
y_train shape: (84429, 5)
X_test shape: 21108
y_test shape: (21108, 5)


In [11]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Ensure the 'Value' column is categorical
double_jeopardy_data['Value'] = pd.Categorical(double_jeopardy_data['Value'])
double_jeopardy_data['Value_cat'] = double_jeopardy_data['Value'].cat.codes

# Define features (X) and target (y)
X_dj = double_jeopardy_data['cleaned_combined_text']
y_dj = to_categorical(double_jeopardy_data['Value_cat'])

# Perform train-test split
X_train_dj, X_test_dj, y_train_dj, y_test_dj = train_test_split(
    X_dj, y_dj, test_size=0.2, random_state=42, stratify=double_jeopardy_data['Value_cat']
)

# Recheck dimensions
print(f"X_train shape: {len(X_train_dj)}")
print(f"y_train shape: {y_train_dj.shape}")
print(f"X_test shape: {len(X_test_dj)}")
print(f"y_test shape: {y_test_dj.shape}")
double_jeopardy_data['Value_cat'] = double_jeopardy_data['Value_cat'].astype(str)

X_train shape: 80907
y_train shape: (80907, 5)
X_test shape: 20227
y_test shape: (20227, 5)


In [12]:
#Jeopardy
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Initialize the tokenizer
tokenizer = Tokenizer(num_words=10000)  # Adjust `num_words` based on the vocabulary size
tokenizer.fit_on_texts(X_train_j)

# Convert text to sequences
X_train_seq_j = tokenizer.texts_to_sequences(X_train_j)
X_test_seq_j = tokenizer.texts_to_sequences(X_test_j)

# Pad sequences to ensure uniform input shape
max_seq_length = 100  # Adjust this based on the typical length of your sequences
X_train_pad_j = pad_sequences(X_train_seq_j, maxlen=max_seq_length, padding='post')
X_test_pad_j = pad_sequences(X_test_seq_j, maxlen=max_seq_length, padding='post')

# Confirm the shapes
print(f"X_train_pad shape: {X_train_pad_j.shape}")
print(f"X_test_pad shape: {X_test_pad_j.shape}")




X_train_pad shape: (84429, 100)
X_test_pad shape: (21108, 100)


In [13]:
max_words = 10000
max_len = 100
# Process a dataset
def preprocess_data(dataset):
    # Encode the Value column as categorical labels
    label_encoder = LabelEncoder()
    dataset['Value'] = label_encoder.fit_transform(dataset['Value'])
    value_categories = label_encoder.classes_  # Save for reference

    # Tokenize and pad sequences
    tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
    tokenizer.fit_on_texts(dataset['cleaned_combined_text'])
    
    sequences = tokenizer.texts_to_sequences(dataset['cleaned_combined_text'])
    padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post', truncating='post')

    # Convert labels to categorical format
    categorical_labels = to_categorical(dataset['Value'])

    return padded_sequences, categorical_labels, value_categories

# Preprocess each dataset
X_jeopardy, y_jeopardy, jeopardy_categories = preprocess_data(jeopardy_data)
X_double_jeopardy, y_double_jeopardy, double_jeopardy_categories = preprocess_data(double_jeopardy_data)

# Split into training and test sets
X_train_jeop, X_test_jeop, y_train_jeop, y_test_jeop = train_test_split(X_jeopardy, y_jeopardy, test_size=0.2, random_state=42)
X_train_dj, X_test_dj, y_train_dj, y_test_dj = train_test_split(X_double_jeopardy, y_double_jeopardy, test_size=0.2, random_state=42)


In [14]:
# Following https://keras.io/examples/nlp/text_classification_with_transformer/
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential(
            [layers.Dense(ff_dim, activation="relu"), layers.Dense(embed_dim),]
        )
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output)
        return self.layernorm2(out1 + ffn_output)

In [15]:
import numpy as np
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = x.shape[-1]
        positions = np.arange(start=0, stop=maxlen, step=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

In [16]:
keras.backend.clear_session()

embed_dim = 64  # Embedding size for each token
num_heads = 2  # Number of attention heads
ff_dim = 32  # Hidden layer size in feed forward network inside transformer
maxlen = 100
vocab_size = 20000

inputs = layers.Input(shape=(maxlen,))
embedding_layer = TokenAndPositionEmbedding(maxlen, vocab_size, embed_dim)
x = embedding_layer(inputs)
transformer_block = TransformerBlock(embed_dim, num_heads, ff_dim)
x = transformer_block(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dropout(0.1)(x)
x = layers.Dense(20, activation="relu")(x)
x = layers.Dropout(0.1)(x)
outputs = layers.Dense(5, activation="softmax")(x)

jeopardy_model = keras.Model(inputs=inputs, outputs=outputs)
double_jeopardy_model = keras.Model(inputs=inputs, outputs=outputs)
# Compile the model with binary crossentropy loss and an adam optimizer. 
# (For whatever reason binary outperforms everything else)

In [17]:
jeopardy_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
# model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

# Training both models
jeopardy_history = jeopardy_model.fit(
    X_train_jeop, y_train_jeop,
    epochs= 3,
    batch_size=64,
    validation_data=(X_test_jeop, y_test_jeop)
)

Epoch 1/3
1320/1320 [==============================] - 17s 12ms/step - loss: 0.5095 - accuracy: 0.2132 - val_loss: 0.4975 - val_accuracy: 0.2439
Epoch 2/3
1320/1320 [==============================] - 15s 11ms/step - loss: 0.4977 - accuracy: 0.2537 - val_loss: 0.4963 - val_accuracy: 0.2515
Epoch 3/3
1320/1320 [==============================] - 15s 11ms/step - loss: 0.4878 - accuracy: 0.2952 - val_loss: 0.5003 - val_accuracy: 0.2430


In [ ]:
double_jeopardy_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
# model.compile(loss="categorical_crossentropy", optimizer="adam", metrics=["accuracy"])

# Training both models
double_jeopardy_history = double_jeopardy_model.fit(
    X_train_dj, y_train_dj,
    epochs= 3,
    batch_size=64,
    validation_data=(X_test_dj, y_test_dj)
)

Epoch 1/3
 999/1265 [======================>.......] - ETA: 2s - loss: 0.4999 - accuracy: 0.2366

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2,1, sharex=True)

ax[0].plot(jeopardy_history.history['loss'])
ax[0].plot(jeopardy_history.history['val_loss'])
ax[0].set_title("Loss")

ax[1].plot(jeopardy_history.history['accuracy'])
ax[1].plot(jeopardy_history.history['val_accuracy'])
ax[1].set_title("Accuracy")




In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2,1, sharex=True)

ax[0].plot(double_jeopardy_history.history['loss'])
ax[0].plot(double_jeopardy_history.history['val_loss'])
ax[0].set_title("Loss")

ax[1].plot(double_jeopardy_history.history['accuracy'])
ax[1].plot(double_jeopardy_history.history['val_accuracy'])
ax[1].set_title("Accuracy")


In [ ]:
jeopardy_predictions = jeopardy_model.predict(X_test_jeop)
jeopardy_predicted_classes = jeopardy_predictions.argmax(axis=1)
jeopardy_true_classes = y_test_jeop.argmax(axis=1)

import pandas as pd

jeopardy_results = pd.DataFrame({
    "True Value": jeopardy_true_classes,
    "Predicted Value": jeopardy_predicted_classes
})
print(jeopardy_results[jeopardy_results['Predicted Value']!=0])

In [ ]:
dj_predictions = double_jeopardy_model.predict(X_test_dj)
dj_predicted_classes = dj_predictions.argmax(axis=1)
dj_true_classes = y_test_dj.argmax(axis=1)

import pandas as pd

double_jeopardy_results = pd.DataFrame({
    "True Value": dj_true_classes,
    "Predicted Value": dj_predicted_classes
})
print(jeopardy_results[jeopardy_results['Predicted Value']!=0])

In [ ]:
confusion_matrix(jeopardy_true_classes, jeopardy_predicted_classes, normalize='true')[1,0]

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

sns.heatmap(confusion_matrix(jeopardy_true_classes, jeopardy_predicted_classes, normalize='true'), annot=True)
plt.ylabel('True')
plt.xlabel('Pred')

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

sns.heatmap(confusion_matrix(dj_true_classes, dj_predicted_classes, normalize='true'), annot=True)
plt.ylabel('True')
plt.xlabel('Pred')